# Model Training (Production Pipeline)
Here we load the featured datasets, remove outliers (only from train), handle the grouped imputation, and run a scikit-learn pipeline.

In [64]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectFromModel

In [65]:
# 1. Load data
train_df = pd.read_csv('Ames_Housing/datasets/train_featured.csv')
test_df = pd.read_csv('Ames_Housing/datasets/test_featured.csv')

# 2. Separate X and y
X_train = train_df.drop('SalePrice', axis=1)
y_train = train_df['SalePrice']

X_test = test_df.drop('SalePrice', axis=1)
y_test = np.log1p(test_df['SalePrice']) # Log transform test target for equivalent evaluation

In [66]:
# 3. Outlier removal on TRAIN only
outlier_mask = (X_train["Gr Liv Area"] > 4000) & (y_train < np.log1p(300000))

X_train = X_train[~outlier_mask]
y_train = y_train[~outlier_mask]
print(f"Removed {outlier_mask.sum()} outliers from training data.")

Removed 2 outliers from training data.


In [67]:
# 4. Custom Grouped Median Imputer
class GroupedMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        
    def fit(self, X, y=None):
        # Learn medians from train
        self.medians_ = X.groupby(self.group_col)[self.target_col].median().to_dict()
        self.global_median_ = X[self.target_col].median()
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.target_col] = X_copy.apply(
            lambda row: self.medians_.get(row[self.group_col], self.global_median_) 
            if pd.isna(row[self.target_col]) else row[self.target_col], axis=1
        )
        return X_copy

# Apply grouped imputer BEFORE the main pipeline
grouped_imputer = GroupedMedianImputer(group_col='Neighborhood', target_col='Lot Frontage')
X_train = grouped_imputer.fit_transform(X_train)
X_test = grouped_imputer.transform(X_test)


In [68]:
# 5. Build Scikit-Learn Pipeline
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ], remainder='passthrough')

# Master Model Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # NEW STEP: Feature Selection using Lasso
    # Lasso will figure out which features are actually useful and drop the rest
    ('feature_selection', SelectFromModel(LassoCV(cv=5))),
    ('regressor', RidgeCV(alphas=(0.1, 1.0, 10.0, 100.0)))
])

In [69]:
# 6. Train & Evaluate!
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"--- Production Model Results ---")
print(f"RMSE (Log Scale): {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

--- Production Model Results ---
RMSE (Log Scale): 0.1227
R2 Score: 0.9186


In [70]:
from sklearn.linear_model import LinearRegression

# 1. Build a new Pipeline that uses LinearRegression instead of Ridge
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor), # Re-use the exact same data cleaning steps!
    # NEW STEP: Feature Selection using Lasso
    # Lasso will figure out which features are actually useful and drop the rest
    ('feature_selection', SelectFromModel(LassoCV(cv=5))), 
    ('regressor', LinearRegression())
])

# 2. Train the model
lr_model.fit(X_train, y_train)

# 3. Predict and Evaluate
lr_y_pred = lr_model.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_y_pred))
lr_r2 = r2_score(y_test, lr_y_pred)

print(f"--- Linear Regression Results ---")
print(f"RMSE (Log Scale): {lr_rmse:.4f}")
print(f"R2 Score: {lr_r2:.4f}")


--- Linear Regression Results ---
RMSE (Log Scale): 0.1232
R2 Score: 0.9179
